# CUDA Aggregation Engine — Colab backend

Runs the FastAPI server that executes **your Numba CUDA kernels** on the Colab GPU and exposes it to the React dashboard through a tunnel.

**Run the cells top to bottom (Runtime → Run all also works after Step 3).** Keep this browser tab open while you use the dashboard.

| Step | Cell | What it does |
|---|---|---|
| 1 | GPU check | Confirms a Tesla T4 is attached |
| 2 | Install | Installs FastAPI, uvicorn, pyngrok (does **not** touch Numba) |
| 3 | Upload | Upload `backend.zip` from the project |
| 4 | Pre-flight | Runs your kernels once on the GPU and checks the answer |
| 5 | Secrets | Reads your ngrok token, creates a fresh API key |
| 6 | Start | Starts FastAPI + ngrok and prints the URL |
**If a cell says `L is not defined` or a variable is missing:** the Colab session was restarted or a step was skipped. Run Steps 3, 4 and 5 again, then Step 6. (Steps 5 and 6 now check this for you.)


## Step 1 · Check the GPU
If this does not show **Tesla T4**: *Runtime → Change runtime type → T4 GPU → Save*, then *Runtime → Restart session*.

In [1]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
from numba import cuda
assert cuda.is_available(), "No GPU attached. Runtime > Change runtime type > T4 GPU, then restart the session."
print("Numba sees:", cuda.get_current_device().name)

name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.82.07
Numba sees: Tesla T4


## Step 2 · Install the web-server packages
Numba, NumPy and pandas are already in Colab — they are intentionally **not** reinstalled.

In [2]:
!pip install -q fastapi uvicorn python-multipart pydantic psutil pyngrok

## Step 3 · Upload the backend
Click **Choose files** and select `backend.zip` (it is in the project folder on your laptop). Re-running this cell replaces the old copy.

In [3]:
import os, shutil, zipfile
from google.colab import files

shutil.rmtree("/content/backend", ignore_errors=True)
up = files.upload()
name = next(iter(up))
assert name.endswith(".zip"), "Please upload backend.zip"
with zipfile.ZipFile(name) as z:
    z.extractall("/content")
assert os.path.exists("/content/backend/app/kernels.py"), "backend.zip must contain a 'backend' folder"
print("Uploaded and extracted to /content/backend")
print(sorted(os.listdir("/content/backend")))

Saving backend.zip to backend.zip
Uploaded and extracted to /content/backend
['app', 'colab_launcher.py', 'requirements.txt', 'run_server.py', 'tests']


## Step 4 · Pre-flight: run your kernels on the GPU
Compiles the six kernels from `backend/app/kernels.py` (copied unchanged from your notebook) and checks them against a known answer (sum of 1…100,000 = 5,000,050,000).
All lines should say **True**.

In [4]:
import sys
sys.path.insert(0, "/content/backend")
import colab_launcher as L

info = L.preflight()
print("GPU:", info["gpu"], "| is Tesla T4:", info["is_t4"], "| Numba", info["numba"])
for c in info["selftest"]:
    print(f'{c["op"]:>4}: expected {c["expected"]:>16,.0f}  GPU {c["gpu"]:>16,.0f}  pass={c["pass"]}')
print("Kernel self-test passed:", info["selftest_passed"])
assert info["selftest_passed"]

/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPe

GPU: Tesla T4 | is Tesla T4: True | Numba 0.64.0
 sum: expected    5,000,050,000  GPU    5,000,050,176  pass=True
 min: expected                1  GPU                1  pass=True
 max: expected          100,000  GPU          100,000  pass=True
Kernel self-test passed: True


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


## Step 5 · Secrets
1. Create a free account at <https://dashboard.ngrok.com/signup> and copy your **authtoken**.
2. In Colab, click the **🔑 Secrets** icon in the left sidebar → *Add new secret* → name `NGROK_AUTHTOKEN`, paste the token → turn on **Notebook access**.
   (If you skip this, the cell asks you to paste it, hidden.)

A new random **API key** is generated every run. Only someone with this key can use your backend.

In [ ]:
import os, sys
if not os.path.exists("/content/backend/colab_launcher.py"):
    raise RuntimeError("The backend files are not in this Colab session (a restart erases them). Run Step 3 (upload backend.zip) first.")
sys.path.insert(0, "/content/backend")
import colab_launcher as L

from getpass import getpass
try:
    from google.colab import userdata
    NGROK_TOKEN = userdata.get("NGROK_AUTHTOKEN")
except Exception:
    NGROK_TOKEN = None
if not NGROK_TOKEN:
    NGROK_TOKEN = getpass("Paste your ngrok authtoken (input is hidden): ").strip()

API_KEY = L.new_api_key()
# The dashboard runs on your laptop at this address. Add more origins separated by commas if you change the port.
ALLOWED_ORIGINS = "http://localhost:5173,http://127.0.0.1:5173"
print("Token loaded:", bool(NGROK_TOKEN), "| API key created (shown in Step 6)")

## Step 6 · Start the server and the tunnel
Prints the **Backend URL** and **API key** — paste both into the dashboard's *Connection* page.

⚠️ Re-run this cell any time Colab disconnects. The URL and key **change** every time.

In [ ]:
import os, sys
if not os.path.exists("/content/backend/colab_launcher.py"):
    raise RuntimeError("The backend files are not in this Colab session (a restart erases them). Run Step 3 (upload backend.zip) first.")
sys.path.insert(0, "/content/backend")
import colab_launcher as L

for _name in ("API_KEY", "NGROK_TOKEN", "ALLOWED_ORIGINS"):
    if _name not in globals():
        raise RuntimeError(f"{_name} is not set. Run Step 5 (Secrets) first, then re-run this cell.")

L.start_server(API_KEY, ALLOWED_ORIGINS)
PUBLIC_URL = L.start_ngrok(NGROK_TOKEN)

check = L.check_public(PUBLIC_URL, API_KEY)      # calls the server through the tunnel, like the browser will
print("Server reachable through the tunnel. GPU:", check["name"], "| warm-up:", check["warmup"])
print()
print("=" * 70)
print("  Backend URL :", PUBLIC_URL)
print("  API key     :", API_KEY)
print("=" * 70)
print("Paste both into the dashboard: Connection page > Save and test connection.")

In [7]:
import os
import socket
import time

print("API key generated:", "API_KEY" in globals())
print("Backend URL generated:", "PUBLIC_URL" in globals())
print("Backend process running:",
      L._state.get("proc") is not None and L._state["proc"].poll() is None)
print("Cloudflared installed:",
      os.path.exists("/usr/local/bin/cloudflared"))

if "PUBLIC_URL" in globals():
    print("Tunnel URL:", PUBLIC_URL)
    host = PUBLIC_URL.replace("https://", "").split("/")[0]
    try:
        print("DNS resolution:", socket.gethostbyname(host))
    except Exception as e:
        print("DNS error:", repr(e))

API key generated: True
Backend URL generated: True
Backend process running: True
Cloudflared installed: True
Tunnel URL: https://cbs-difficulties-validation-playstation.trycloudflare.com
DNS resolution: 104.16.230.132


In [8]:
print("Backend URL:", PUBLIC_URL)

try:
    check = L.check_public(PUBLIC_URL, API_KEY)
    print("Public connection: SUCCESS")
    print("GPU:", check["name"])
    print("Warm-up:", check["warmup"])
except Exception as e:
    print("Public connection failed:", repr(e))

Backend URL: https://cbs-difficulties-validation-playstation.trycloudflare.com
Public connection: SUCCESS
GPU: Tesla T4
Warm-up: {'ok': True, 'error': None}


In [10]:
print("API KEY:", API_KEY)

API KEY: QTry16TywRJsYpHt2USZwh3xoNP36oGP


In [ ]:
import os
import sys

sys.path.insert(0, "/content/backend")
import colab_launcher as L

# Generate a fresh API key
API_KEY = L.new_api_key()

# Allow your local React frontend
ALLOWED_ORIGINS = "http://localhost:5173,http://127.0.0.1:5173"

# Start FastAPI backend
L.start_server(API_KEY, ALLOWED_ORIGINS)

# Create a Cloudflare tunnel instead of ngrok
PUBLIC_URL = L.start_cloudflared()

# Verify public connectivity
check = L.check_public(PUBLIC_URL, API_KEY)

print("Connected successfully!")
print("GPU:", check["name"])
print()
print("=" * 60)
print("Backend URL:", PUBLIC_URL)
print("API KEY:", API_KEY)
print("=" * 60)

## Server log (run any time)
Useful if something fails. Each request from the dashboard appears here.

In [ ]:
import os, sys
if not os.path.exists("/content/backend/colab_launcher.py"):
    raise RuntimeError("The backend files are not in this Colab session (a restart erases them). Run Step 3 (upload backend.zip) first.")
sys.path.insert(0, "/content/backend")
import colab_launcher as L

print(L.tail_log(40))

## Optional · No ngrok account? Use a Cloudflare quick tunnel
Run **instead of** the ngrok part of Step 6 (keep `L.start_server(...)` first). The URL looks like `https://something.trycloudflare.com`.

In [ ]:
# L.start_server(API_KEY, ALLOWED_ORIGINS)
# PUBLIC_URL = L.start_cloudflared()
# print(PUBLIC_URL, API_KEY)

## Stop everything
Run this when you are finished (also frees the GPU).

In [ ]:
import os, sys
if not os.path.exists("/content/backend/colab_launcher.py"):
    raise RuntimeError("The backend files are not in this Colab session (a restart erases them). Run Step 3 (upload backend.zip) first.")
sys.path.insert(0, "/content/backend")
import colab_launcher as L

L.stop_tunnels(); L.stop_server(); print('Stopped.')

**Your Cloudflare startup cell**

In [ ]:
import secrets

# Generate a fresh API key
API_KEY = L.new_api_key()

# Allow your local React frontend
ALLOWED_ORIGINS = "http://localhost:5173,http://127.0.0.1:5173"

# Start the FastAPI backend
L.start_server(API_KEY, ALLOWED_ORIGINS)

# Start Cloudflare Quick Tunnel
PUBLIC_URL = L.start_cloudflared()

# Verify public connection
check = L.check_public(PUBLIC_URL, API_KEY)

print("Backend successfully connected!")
print("GPU:", check["name"])
print("Warm-up:", check["warmup"])
print()
print("Backend URL:", PUBLIC_URL)
print("API KEY:", API_KEY)